# 서울시 노인 이동·사고·환승 부담 종합 시각화

## 데이터 결합
1. **TAAS 노인 보행사고** (서울 전체 2024) — `data/processed/taas_seoul_elderly_pedestrian_2024.csv`
2. **노인 카드 trip** — `elderly_card_trips` (505K, 2026-02-22 일요일)
3. **환승 접근성** — `monthly_transfer_accessibility` (2025-10)
4. **정류소·역 좌표** — `stop_coords` + `station_coords` (수도권 ~35K)

## 핵심 지표
- **노인 승차/하차 정류소** — 빈도 기반 핫스팟
- **환승 부담 정류소** — 평균 환승시간(trnf_hr) 긴 곳
- **결합 위험도** — 노인 이용 ↑ × 환승 시간 ↑ = 환승 취약 정류소
- **사고 인접 정류소** — 200m 반경 사고 매칭 (BallTree)

## 시각화
Kakao Map HTML — 서울 25구 인터랙티브
- 레이어: 사고 / 노인 승차 / 노인 하차 / 환승 부담 / 사고 인접 정류소
- 필터: 구 / 시간대 / 환승 유형

## 단계
1. 환경 로드
2. 통합 view `sttn_coords` 생성 (sttn_id → 좌표)
3. 노인 trip 정류소 집계
4. 환승 부담 집계
5. 결합 위험 지표
6. 사고 인접 정류소 매칭 (BallTree)
7. Kakao Map HTML

(횡단보도는 다음 단계)

In [15]:
# ============================================================
# 셀 1 - 환경 + DB 헬퍼 + TAAS CSV 로드
# ============================================================
import duckdb
import pandas as pd
import numpy as np
import json
from pathlib import Path
from contextlib import contextmanager
from sklearn.neighbors import BallTree

DB_PATH = "data/seoul.duckdb"

@contextmanager
def db_open(read_only: bool = False):
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()

# TAAS 사고 데이터 로드
accidents = pd.read_csv("data/processed/taas_seoul_elderly_pedestrian_2024.csv")
accidents['구'] = accidents['legaldong_name'].str.extract(r'서울특별시 (\S+구)')[0]
accidents = accidents.dropna(subset=['lat', 'lon', '구']).reset_index(drop=True)

print(f"TAAS 사고: {len(accidents):,}건 (구별 분포)")
print(accidents['구'].value_counts().to_string())

TAAS 사고: 1,963건 (구별 분포)
구
동대문구    130
송파구     116
은평구     111
강남구     110
성북구     103
강동구     100
구로구      97
강북구      94
강서구      92
중랑구      88
양천구      87
노원구      85
관악구      82
동작구      82
영등포구     79
서대문구     69
중구       61
도봉구      61
종로구      60
금천구      54
광진구      47
서초구      47
마포구      38
성동구      37
용산구      33


In [16]:
# ============================================================
# 셀 2 - 통합 view: sttn_id → 좌표 (버스 + 지하철)
# ============================================================
# bus: route_station × stop_coords (key: ctpv+sgg+sttn_nm)
# subway: railway_station × station_coords (key: sttn_nm)
# 양쪽 다 sttn_id로 lookup 가능하게 만듦

with db_open() as con:
    con.execute("""
    CREATE OR REPLACE VIEW sttn_coords AS
    -- 버스 정류소 (서울)
    SELECT DISTINCT
        rs.sttn_id,
        rs.sttn_nm,
        rs.ctpv_cd,
        rs.sgg_cd,
        sc.lat,
        sc.lon,
        'bus' AS kind
    FROM route_station rs
    LEFT JOIN stop_coords sc
      ON sc.ctpv_cd = rs.ctpv_cd
     AND sc.sgg_cd  = rs.sgg_cd
     AND sc.sttn_nm = rs.sttn_nm
    WHERE rs.ctpv_cd = '11'
    
    UNION ALL
    
    -- 지하철역 (수도권)
    SELECT DISTINCT
        ws.sttn_id,
        ws.sttn_nm,
        '11' AS ctpv_cd,
        NULL AS sgg_cd,
        stc.lat,
        stc.lon,
        'subway' AS kind
    FROM railway_station ws
    LEFT JOIN station_coords stc
      ON stc.sttn_nm = ws.sttn_nm
     AND stc.ctpv_cd = '11'
    WHERE ws.sarea_nm = '수도권'
    """)

with db_open(read_only=True) as con:
    print("=== sttn_coords 구성 ===")
    print(con.execute("""
        SELECT kind, 
               COUNT(*) AS rows,
               COUNT(DISTINCT sttn_id) AS unique_sttn,
               SUM(CASE WHEN lat IS NOT NULL THEN 1 ELSE 0 END) AS 좌표보유,
               ROUND(100.0*SUM(CASE WHEN lat IS NOT NULL THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct
        FROM sttn_coords GROUP BY kind
    """).df().to_string())
    
    print("\n=== card_trip 정류소 좌표 매칭률 ===")
    print(con.execute("""
        WITH used AS (
            SELECT DISTINCT ride_sttn_id AS sttn_id FROM elderly_card_trips WHERE ride_sttn_id IS NOT NULL
            UNION SELECT DISTINCT goff_sttn_id FROM elderly_card_trips WHERE goff_sttn_id IS NOT NULL
        )
        SELECT
            COUNT(*) AS total,
            COUNT(sc.lat) AS matched,
            ROUND(100.0*COUNT(sc.lat)/COUNT(*), 1) AS pct
        FROM used u LEFT JOIN sttn_coords sc ON sc.sttn_id = u.sttn_id
    """).df().to_string())

=== sttn_coords 구성 ===
     kind   rows  unique_sttn     좌표보유    pct
0  subway    784          784    784.0  100.0
1     bus  15748        15745  15283.0   97.0

=== card_trip 정류소 좌표 매칭률 ===
   total  matched   pct
0  12492    11026  88.3


In [17]:
# ============================================================
# 셀 3 - 노인 trip 정류소 집계 (승차 / 하차 / 시간대별)
# ============================================================
with db_open(read_only=True) as con:
    # 승차 정류소 — sttn_id 기준 (서울 좌표 매칭된 것만)
    ride_agg = con.execute("""
        WITH agg AS (
            SELECT ride_sttn_id AS sttn_id,
                   COUNT(*) AS ride_cnt,
                   COUNT(DISTINCT vr_card_no) AS 고유_노인,
                   ROUND(AVG(utztn_dstnc), 0) AS 평균이동거리_m
            FROM elderly_card_trips
            WHERE ride_sttn_id IS NOT NULL
            GROUP BY ride_sttn_id
        )
        SELECT a.sttn_id, sc.sttn_nm, sc.ctpv_cd, sc.sgg_cd, sc.kind,
               sc.lat, sc.lon,
               a.ride_cnt, a.고유_노인, a.평균이동거리_m
        FROM agg a
        LEFT JOIN sttn_coords sc ON sc.sttn_id = a.sttn_id
        WHERE sc.lat IS NOT NULL  -- 좌표 있는 것만
    """).df()

    goff_agg = con.execute("""
        WITH agg AS (
            SELECT goff_sttn_id AS sttn_id,
                   COUNT(*) AS goff_cnt
            FROM elderly_card_trips
            WHERE goff_sttn_id IS NOT NULL
            GROUP BY goff_sttn_id
        )
        SELECT a.sttn_id, sc.sttn_nm, sc.ctpv_cd, sc.sgg_cd, sc.kind,
               sc.lat, sc.lon,
               a.goff_cnt
        FROM agg a
        LEFT JOIN sttn_coords sc ON sc.sttn_id = a.sttn_id
        WHERE sc.lat IS NOT NULL
    """).df()

print(f"승차 정류소(좌표있음): {len(ride_agg):,}개, 총 trip {ride_agg['ride_cnt'].sum():,}건")
print(f"하차 정류소(좌표있음): {len(goff_agg):,}개, 총 trip {goff_agg['goff_cnt'].sum():,}건")

print("\n=== 노인 승차 TOP 15 정류소 (서울) ===")
print(ride_agg.nlargest(15, 'ride_cnt')[['sttn_nm', 'kind', 'ride_cnt', '고유_노인', '평균이동거리_m', 'lat', 'lon']].to_string())

승차 정류소(좌표있음): 7,901개, 총 trip 505,381건
하차 정류소(좌표있음): 8,843개, 총 trip 500,746건

=== 노인 승차 TOP 15 정류소 (서울) ===
                 sttn_nm    kind  ride_cnt  고유_노인  평균이동거리_m        lat         lon
5427           영등포 [1호선]  subway      6831   5278   11084.0  37.516568  126.907065
2961  청량리(서울시립대입구) [1호선]  subway      5690   4855    9863.0  37.580408  127.045503
2500         고속터미널 [3호선]  subway      5433   4413    9291.0  37.503544  127.006304
3491           연신내 [3호선]  subway      5048   3879    8147.0  37.619327  126.920945
5966      잠실(송파구청) [2호선]  subway      4560   3306   10033.0  37.513843  127.099632
4902          종로3가 [1호선]  subway      4553   3800   10510.0  37.570641  126.993204
2502            창동 [4호선]  subway      4517   3606    6932.0  37.653577  127.049973
524            신도림 [2호선]  subway      4213   3487    8978.0  37.507629  126.891880
4438            서울 [1호선]  subway      4163   3406   10451.0  37.567509  126.977369
6862           제기동 [1호선]  subway      4148   3677    9940.0  37

In [18]:
# ============================================================
# 셀 4 - 환승 부담 정류소 집계
# ============================================================
# 컬럼 의미:
#   trnf_hr   = SUM 환승시간(초)  ← 분자
#   pasg_cnt  = 환승 인원수       ← 분모
# ⇒ 평균 환승시간 = SUM(trnf_hr) / SUM(pasg_cnt)
# trnf_type_cd: BB(버스→버스), BT(버스→지하철), TB(지하철→버스), TT(지하철→지하철)

with db_open(read_only=True) as con:
    transfer_agg = con.execute("""
        WITH agg AS (
            SELECT sttn_id,
                   SUM(pasg_cnt) AS 총환승인원,
                   ROUND(SUM(trnf_hr * 1.0) / NULLIF(SUM(pasg_cnt), 0), 0) AS 평균환승시간_초,
                   COUNT(DISTINCT trnf_type_cd) AS 환승유형수
            FROM monthly_transfer_accessibility
            WHERE pasg_cnt > 0
            GROUP BY sttn_id
        )
        SELECT a.sttn_id, sc.sttn_nm, sc.ctpv_cd, sc.sgg_cd, sc.kind,
               sc.lat, sc.lon,
               a.총환승인원, a.평균환승시간_초, a.환승유형수
        FROM agg a
        LEFT JOIN sttn_coords sc ON sc.sttn_id = a.sttn_id
        WHERE sc.lat IS NOT NULL
    """).df()

print(f"환승 정류소(좌표있음): {len(transfer_agg):,}개")
print(f"\n환승 시간 분위수:")
print(transfer_agg['평균환승시간_초'].describe().round(0).to_string())

print("\n=== 환승 시간 가장 긴 정류소 TOP 15 (인원 1000명+) ===")
high_burden = transfer_agg[transfer_agg['총환승인원'] >= 1000].nlargest(15, '평균환승시간_초')
print(high_burden[['sttn_nm', 'kind', '총환승인원', '평균환승시간_초', 'lat', 'lon']].to_string())

환승 정류소(좌표있음): 14,900개

환승 시간 분위수:
count    14900.0
mean       738.0
std        316.0
min         22.0
25%        532.0
50%        669.0
75%        885.0
max       8976.0

=== 환승 시간 가장 긴 정류소 TOP 15 (인원 1000명+) ===
             sttn_nm    kind   총환승인원  평균환승시간_초        lat         lon
2929        서울 [1호선]  subway  1411.0    2237.0  37.567509  126.977369
12264         남산서울타워     bus  1066.0    1545.0  37.550967  126.991157
4651             농협앞     bus  1001.0    1280.0  37.524053  127.044370
13197        한남동주민센터     bus  1433.0    1259.0  37.534062  127.000591
9466       서울 [공항철도]  subway  1987.0    1203.0  37.553152  126.968882
4144           홍익대학교     bus  1224.0    1187.0  37.551221  126.927337
6974        홍대입구역사거리     bus  1164.0    1166.0  37.554675  126.921167
5482   금천구종합청사.금천구청역     bus  1506.0    1153.0  37.456116  126.894520
4748   퇴계로2가.명동역5번출구     bus  1545.0    1143.0  37.560920  126.984488
8140            공릉시장     bus  1288.0    1141.0  37.622811  127.074112
12452       반포미도아

In [19]:
# ============================================================
# 셀 5 - 결합 지표: 노인 이용 × 환승 부담
# ============================================================
# z-score 정규화 후 합한 값으로 위험도 산출
# 평균 환승시간 = SUM(trnf_hr) / SUM(pasg_cnt)

with db_open(read_only=True) as con:
    combined = con.execute("""
        WITH ride AS (
            SELECT ride_sttn_id AS sttn_id, COUNT(*) AS ride_cnt
            FROM elderly_card_trips WHERE ride_sttn_id IS NOT NULL GROUP BY ride_sttn_id
        ),
        goff AS (
            SELECT goff_sttn_id AS sttn_id, COUNT(*) AS goff_cnt
            FROM elderly_card_trips WHERE goff_sttn_id IS NOT NULL GROUP BY goff_sttn_id
        ),
        trans AS (
            SELECT sttn_id,
                   SUM(pasg_cnt) AS 총환승인원,
                   SUM(trnf_hr * 1.0) / NULLIF(SUM(pasg_cnt), 0) AS 평균환승시간_초
            FROM monthly_transfer_accessibility
            WHERE pasg_cnt > 0
            GROUP BY sttn_id
        ),
        joined AS (
            SELECT 
                COALESCE(ride.sttn_id, goff.sttn_id, trans.sttn_id) AS sttn_id,
                COALESCE(ride_cnt, 0) AS ride_cnt,
                COALESCE(goff_cnt, 0) AS goff_cnt,
                COALESCE(ride_cnt, 0) + COALESCE(goff_cnt, 0) AS 노인총이용,
                COALESCE(총환승인원, 0) AS 총환승인원,
                COALESCE(평균환승시간_초, 0) AS 평균환승시간_초
            FROM ride
            FULL JOIN goff USING (sttn_id)
            FULL JOIN trans USING (sttn_id)
        )
        SELECT j.sttn_id, sc.sttn_nm, sc.ctpv_cd, sc.sgg_cd, sc.kind,
               sc.lat, sc.lon,
               j.ride_cnt, j.goff_cnt, j.노인총이용, j.총환승인원, j.평균환승시간_초
        FROM joined j
        LEFT JOIN sttn_coords sc ON sc.sttn_id = j.sttn_id
        WHERE sc.lat IS NOT NULL
          AND j.노인총이용 > 0
    """).df()

# 위험도 계산 — z-score
from scipy.stats import zscore
combined = combined[combined['평균환승시간_초'] > 0].copy()
combined['z_노인이용'] = zscore(np.log1p(combined['노인총이용']))
combined['z_환승시간'] = zscore(combined['평균환승시간_초'])
combined['위험도'] = combined['z_노인이용'] + combined['z_환승시간']

print(f"결합 분석 정류소: {len(combined):,}개")
print("\n=== 노인 환승 부담 핫스팟 TOP 20 (위험도 = 노인이용 z + 환승시간 z) ===")
print(combined.nlargest(20, '위험도')[
    ['sttn_nm', 'kind', '노인총이용', '평균환승시간_초', 'z_노인이용', 'z_환승시간', '위험도']
].round(2).to_string())

결합 분석 정류소: 10,659개

=== 노인 환승 부담 핫스팟 TOP 20 (위험도 = 노인이용 z + 환승시간 z) ===
                sttn_nm    kind  노인총이용  평균환승시간_초  z_노인이용  z_환승시간    위험도
4954            한화비즈메트로     bus      1   5087.16   -0.99   15.50  14.51
3956                빨래터     bus      1   3822.95   -0.99   11.00  10.01
4720        우리옛돌박물관.정법사     bus      6   3447.00   -0.10    9.66   9.56
3188               새원마을     bus      1   3606.71   -0.99   10.23   9.24
5424      아카데미하우스.통일교육원     bus      3   3208.80   -0.50    8.82   8.32
5995              성북구의회     bus      1   3242.70   -0.99    8.94   7.94
10389          서울 [1호선]  subway    200   2237.40    2.28    5.36   7.64
3948              승가사입구     bus      2   2969.63   -0.71    7.97   7.26
10190             방배중학교     bus     10   2600.36    0.22    6.65   6.87
1440            한신108동앞     bus      8   2361.19    0.08    5.80   5.88
7751    서울대학교병원현관.암병원현관     bus     36   2045.00    1.08    4.67   5.76
2109              골드하우스     bus      4   2423.80   -0.34    6.02

In [20]:
# ============================================================
# 셀 6 - 사고 인접 정류소 매칭 (BallTree, 200m)
# ============================================================
# 노인 이용 정류소 좌표 ↔ TAAS 사고 좌표
# 정류소 200m 반경에 사고 N건 → 위험 정류소

stop_pts = combined[['lat', 'lon']].values
acc_pts  = accidents[['lat', 'lon']].values

EARTH_R = 6371000
RADIUS_M = 200

stop_rad = np.radians(stop_pts)
acc_rad  = np.radians(acc_pts)

tree_acc = BallTree(acc_rad, metric='haversine')
near_acc = tree_acc.query_radius(stop_rad, r=RADIUS_M / EARTH_R, count_only=True)
combined['주변사고수_200m'] = near_acc

print(f"평균 주변 사고: {combined['주변사고수_200m'].mean():.2f}건")
print(f"최대 주변 사고: {combined['주변사고수_200m'].max()}건")
print(f"사고 인접 정류소 (1건 이상): {(combined['주변사고수_200m']>0).sum():,}개 / {len(combined):,}")

# 종합 위험도: 노인 이용 + 환승시간 + 사고밀집
combined['z_사고'] = zscore(np.log1p(combined['주변사고수_200m']))
combined['종합위험도'] = combined['z_노인이용'] + combined['z_환승시간'] + combined['z_사고']

print("\n=== 노인 종합 위험 정류소 TOP 20 (이용 + 환승 + 사고) ===")
print(combined.nlargest(20, '종합위험도')[
    ['sttn_nm', 'kind', '노인총이용', '평균환승시간_초', '주변사고수_200m', '종합위험도']
].round(2).to_string())

평균 주변 사고: 0.93건
최대 주변 사고: 21건
사고 인접 정류소 (1건 이상): 5,281개 / 10,659

=== 노인 종합 위험 정류소 TOP 20 (이용 + 환승 + 사고) ===
                  sttn_nm    kind  노인총이용  평균환승시간_초  주변사고수_200m  종합위험도
4954              한화비즈메트로     bus      1   5087.16           2  15.62
3956                  빨래터     bus      1   3822.95           0   9.13
10389            서울 [1호선]  subway    200   2237.40           2   8.76
4720          우리옛돌박물관.정법사     bus      6   3447.00           0   8.69
3188                 새원마을     bus      1   3606.71           0   8.36
5424        아카데미하우스.통일교육원     bus      3   3208.80           0   7.44
8474          청량리.청과물도매시장     bus    214    795.71          21   7.28
10619           영등포 [1호선]  subway  11442    562.72           6   7.20
5995                성북구의회     bus      1   3242.70           0   7.07
2713                경동시장앞     bus     85    904.54          21   7.01
10476           동묘앞 [1호선]  subway   6685    513.26           6   6.65
5329                 경동시장     bus     97    755.94 

In [21]:
# ============================================================
# 셀 7 - Kakao Map HTML 생성 (서울 25구 인터랙티브)
# ============================================================
kakao_key = "4827d1df867dfc08ae1daba2b1d25835"

# 사고 데이터
gu_list = sorted(accidents['구'].unique().tolist())
gu_counts = accidents['구'].value_counts().to_dict()
vhcle_counts = accidents['wrngdo_vhcle_asort_dc'].fillna('불명').value_counts()

all_accident = accidents[
    ['lat','lon','구','legaldong_name','acdnt_dd_dc','acdnt_gae_dc','road_stle_dc','wrngdo_vhcle_asort_dc']
].dropna(subset=['lat','lon']).values.tolist()

# 정류소 데이터 (sgg_cd → 구 이름 변환 위해 region에서 가져옴)
with db_open(read_only=True) as con:
    sgg_map = con.execute("""
        SELECT DISTINCT sido_cd || sgg_cd AS sgg_full,
               regexp_replace(locatadd_nm, '^서울특별시 ', '') AS gu_nm
        FROM region
        WHERE sido_cd='11' AND sgg_cd<>'000' AND umd_cd='000' AND ri_cd='00'
    """).df()
sgg_to_gu = dict(zip(sgg_map['sgg_full'], sgg_map['gu_nm']))

combined['gu_nm'] = combined['sgg_cd'].map(sgg_to_gu).fillna('지하철역')

# 핵심 정류소만 표시 (위험도 상위 또는 노인 이용 많은 곳)
TOP_RIDE = 200      # 노인 승차 TOP
TOP_TRANS = 200     # 환승 부담 TOP
TOP_RISK = 200      # 종합 위험 TOP
TOP_ACC = 200       # 사고 인접 TOP

ride_top = combined.nlargest(TOP_RIDE, '노인총이용')[
    ['lat','lon','sttn_nm','kind','gu_nm','노인총이용','ride_cnt','goff_cnt']
].values.tolist()
trans_top = combined[combined['총환승인원']>=500].nlargest(TOP_TRANS, '평균환승시간_초')[
    ['lat','lon','sttn_nm','kind','gu_nm','평균환승시간_초','총환승인원']
].values.tolist()
risk_top = combined.nlargest(TOP_RISK, '종합위험도')[
    ['lat','lon','sttn_nm','kind','gu_nm','종합위험도','노인총이용','평균환승시간_초','주변사고수_200m']
].values.tolist()
acc_near_top = combined[combined['주변사고수_200m']>=2].nlargest(TOP_ACC, '주변사고수_200m')[
    ['lat','lon','sttn_nm','kind','gu_nm','주변사고수_200m','노인총이용']
].values.tolist()

gu_options = '\n'.join([f'        <option value="{g}">{g} ({gu_counts.get(g,0)}건)</option>' for g in gu_list])
vhcle_options = '\n'.join([f'        <option value="{v}">{v} ({c})</option>' for v,c in vhcle_counts.items()])

html = f"""<!DOCTYPE html>
<html><head>
<meta charset="utf-8">
<title>서울 노인 보행/이동/환승 종합 위험 지도</title>
<style>
* {{ margin: 0; padding: 0; box-sizing: border-box; }}
body {{ font-family: 'Apple SD Gothic Neo', -apple-system, sans-serif; }}
#map {{ width: 100vw; height: 100vh; }}
#control {{
    position: absolute; top: 16px; left: 16px;
    background: white; padding: 14px 16px;
    border-radius: 12px; box-shadow: 0 4px 16px rgba(0,0,0,0.2);
    font-size: 13px; z-index: 10; min-width: 280px;
    max-height: calc(100vh - 32px); overflow-y: auto;
}}
h4 {{ font-size: 14px; margin-bottom: 10px; padding-bottom: 6px; border-bottom: 2px solid #e74c3c; }}
.group {{ margin-bottom: 10px; padding-bottom: 8px; border-bottom: 1px solid #eee; }}
.group:last-child {{ border-bottom: none; }}
.group-label {{ font-size: 11px; color: #888; margin-bottom: 4px; font-weight: 600; }}
.item {{ display: flex; align-items: center; margin-bottom: 5px; cursor: pointer; user-select: none; }}
.item input {{ margin-right: 7px; }}
.dot {{ display: inline-block; width: 11px; height: 11px; border-radius: 50%; margin-right: 6px; flex-shrink: 0; }}
select {{ width: 100%; padding: 5px 8px; border: 1px solid #ddd; border-radius: 6px; font-size: 13px; }}
#info {{ position: absolute; bottom: 16px; left: 16px;
    background: white; padding: 10px 14px; border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15); font-size: 12px; color: #555; z-index: 10; }}
</style>
<script src="//dapi.kakao.com/v2/maps/sdk.js?appkey={kakao_key}&libraries=services,clusterer"></script>
</head>
<body>
<div id="map"></div>
<div id="control">
<h4>🚸 서울 노인 위험 종합 지도</h4>
<div class="group">
<div class="group-label">구 선택</div>
<select id="guFilter"><option value="all">서울 전체 ({len(all_accident)}건)</option>
{gu_options}
</select></div>
<div class="group">
<div class="group-label">가해차량</div>
<select id="vhcleFilter"><option value="all">전체</option>
{vhcle_options}
</select></div>
<div class="group"><div class="group-label">사고</div>
<div class="item"><input type="checkbox" id="toggleAcc" checked>
<span class="dot" style="background:#f39c12"></span>노인 보행사고 (<span id="accCnt">{len(all_accident)}</span>건)</div>
<div class="item"><input type="checkbox" id="toggleHeat" checked>
<span class="dot" style="background:rgba(255,100,0,.6); border-radius:2px"></span>사고 히트맵</div>
</div>
<div class="group"><div class="group-label">노인 카드 trip 정류소</div>
<div class="item"><input type="checkbox" id="toggleRide">
<span class="dot" style="background:#3498db"></span>승차 핫스팟 ({len(ride_top)})</div>
</div>
<div class="group"><div class="group-label">환승 부담</div>
<div class="item"><input type="checkbox" id="toggleTrans">
<span class="dot" style="background:#9b59b6"></span>환승 시간 긴 정류소 ({len(trans_top)})</div>
</div>
<div class="group"><div class="group-label">위험 정류소 (결합)</div>
<div class="item"><input type="checkbox" id="toggleRisk" checked>
<span class="dot" style="background:#e74c3c"></span>종합 위험도 TOP ({len(risk_top)})</div>
<div class="item"><input type="checkbox" id="toggleAccNear">
<span class="dot" style="background:#c0392b"></span>사고 인접 정류소 ({len(acc_near_top)})</div>
</div>
</div>
<div id="info"><span id="infoText">서울 노인 보행사고 {len(all_accident)}건 · 노인 trip 50.5만 · 환승 분석 결합</span></div>
<script>
var map = new kakao.maps.Map(document.getElementById('map'), {{
    center: new kakao.maps.LatLng(37.5665, 126.9780), level: 8
}});
var allAcc = {json.dumps(all_accident, ensure_ascii=False)};
var rideTop = {json.dumps(ride_top, ensure_ascii=False)};
var transTop = {json.dumps(trans_top, ensure_ascii=False)};
var riskTop = {json.dumps(risk_top, ensure_ascii=False)};
var accNearTop = {json.dumps(acc_near_top, ensure_ascii=False)};

var accOverlays=[], heatCircles=[], rideOverlays=[], transOverlays=[], riskOverlays=[], accNearOverlays=[];
var currentGu='all', currentVhcle='all';

function makeDot(opts) {{
    var el = document.createElement('div');
    Object.assign(el.style, opts);
    return el;
}}

// 사고 마커
allAcc.forEach(function(d){{
    var lat=d[0], lon=d[1], gu=d[2], dong=d[3], date=d[4], severity=d[5], road=d[6], vhcle=d[7]||'불명';
    var el = makeDot({{width:'10px',height:'10px',background:'#f39c12',borderRadius:'50%',
        border:'1px solid white',cursor:'pointer',boxShadow:'0 1px 4px rgba(0,0,0,.3)'}});
    el.title = `[${{gu}}] ${{dong}}\\n${{date}} · ${{severity}}\\n${{road}} · ${{vhcle}}`;
    var ov = new kakao.maps.CustomOverlay({{
        position: new kakao.maps.LatLng(lat,lon), content: el, yAnchor: 0.5, xAnchor: 0.5
    }});
    accOverlays.push({{ ov: ov, gu: gu, vhcle: vhcle }});
    var c = new kakao.maps.Circle({{
        center: new kakao.maps.LatLng(lat,lon), radius: 50,
        strokeWeight: 0, fillColor: '#ff6400', fillOpacity: 0.15
    }});
    heatCircles.push({{ c: c, gu: gu, vhcle: vhcle }});
}});

// 정류소 오버레이 빌더 (radius=노인이용 비례)
function makeStopOverlays(data, color, radiusFn, labelFn, store) {{
    data.forEach(function(d){{
        var lat=d[0], lon=d[1];
        var r = radiusFn(d);
        var el = makeDot({{width: r+'px', height: r+'px', background: color,
            borderRadius:'50%', border:'2px solid white', opacity:'0.7', cursor:'pointer',
            boxShadow:'0 2px 6px rgba(0,0,0,.3)'}});
        el.title = labelFn(d);
        var ov = new kakao.maps.CustomOverlay({{
            position: new kakao.maps.LatLng(lat,lon), content: el, yAnchor: 0.5, xAnchor: 0.5
        }});
        store.push({{ ov: ov, gu: d[4] }});
    }});
}}

makeStopOverlays(rideTop, '#3498db',
    d => Math.max(8, Math.min(30, Math.sqrt(d[5])*0.5)),
    d => `${{d[2]}} (${{d[4]}}) · ${{d[3]}}\\n노인 이용 ${{d[5]}}건`,
    rideOverlays);

makeStopOverlays(transTop, '#9b59b6',
    d => Math.max(10, Math.min(28, d[5]/30)),
    d => `${{d[2]}} (${{d[4]}})\\n평균 환승 ${{Math.round(d[5])}}초 · ${{d[6]}}명`,
    transOverlays);

makeStopOverlays(riskTop, '#e74c3c',
    d => Math.max(12, Math.min(34, 8 + d[5]*4)),
    d => `${{d[2]}} (${{d[4]}})\\n위험도 ${{d[5].toFixed(2)}}\\n노인 ${{d[6]}}건 · 환승 ${{Math.round(d[7])}}초 · 사고 ${{d[8]}}건`,
    riskOverlays);

makeStopOverlays(accNearTop, '#c0392b',
    d => Math.max(10, Math.min(28, 8 + d[5]*2)),
    d => `${{d[2]}} (${{d[4]}})\\n주변 사고 ${{d[5]}}건 · 노인 ${{d[6]}}건`,
    accNearOverlays);

// 토글 + 필터 적용
function apply() {{
    var aOn = document.getElementById('toggleAcc').checked;
    var hOn = document.getElementById('toggleHeat').checked;
    var rOn = document.getElementById('toggleRide').checked;
    var tOn = document.getElementById('toggleTrans').checked;
    var kOn = document.getElementById('toggleRisk').checked;
    var nOn = document.getElementById('toggleAccNear').checked;
    
    var visAcc = 0;
    accOverlays.forEach(function(x){{
        var pass = aOn && (currentGu==='all' || x.gu===currentGu) && (currentVhcle==='all' || x.vhcle===currentVhcle);
        x.ov.setMap(pass ? map : null);
        if (pass) visAcc++;
    }});
    heatCircles.forEach(function(x){{
        var pass = hOn && (currentGu==='all' || x.gu===currentGu) && (currentVhcle==='all' || x.vhcle===currentVhcle);
        x.c.setMap(pass ? map : null);
    }});
    rideOverlays.forEach(function(x){{ x.ov.setMap(rOn && (currentGu==='all'||x.gu===currentGu) ? map : null); }});
    transOverlays.forEach(function(x){{ x.ov.setMap(tOn && (currentGu==='all'||x.gu===currentGu) ? map : null); }});
    riskOverlays.forEach(function(x){{ x.ov.setMap(kOn && (currentGu==='all'||x.gu===currentGu) ? map : null); }});
    accNearOverlays.forEach(function(x){{ x.ov.setMap(nOn && (currentGu==='all'||x.gu===currentGu) ? map : null); }});
    
    document.getElementById('accCnt').textContent = visAcc;
    document.getElementById('infoText').textContent = 
        `${{currentGu==='all'?'서울 전체':currentGu}} · 사고 ${{visAcc}}건` +
        (currentVhcle!=='all' ? ` · ${{currentVhcle}}` : '');
}}

['toggleAcc','toggleHeat','toggleRide','toggleTrans','toggleRisk','toggleAccNear'].forEach(function(id){{
    document.getElementById(id).addEventListener('change', apply);
}});
document.getElementById('guFilter').addEventListener('change', function(){{
    currentGu = this.value;
    if (currentGu !== 'all') {{
        var pts = accOverlays.filter(function(x){{return x.gu===currentGu;}}).map(function(x){{return x.ov.getPosition();}});
        if (pts.length) {{
            var bounds = new kakao.maps.LatLngBounds();
            pts.forEach(function(p){{ bounds.extend(p); }});
            map.setBounds(bounds);
        }}
    }} else {{
        map.setCenter(new kakao.maps.LatLng(37.5665, 126.9780)); map.setLevel(8);
    }}
    apply();
}});
document.getElementById('vhcleFilter').addEventListener('change', function(){{
    currentVhcle = this.value; apply();
}});

apply();
</script>
</body></html>
"""

Path('seoul_elderly_map.html').write_text(html, encoding='utf-8')
print(f"✅ 생성: seoul_elderly_map.html")
print(f"   사고: {len(all_accident):,}건")
print(f"   승차 TOP: {len(ride_top)}, 환승 TOP: {len(trans_top)}, 위험 TOP: {len(risk_top)}, 사고인접: {len(acc_near_top)}")
print(f"\n브라우저에서 열기: open seoul_elderly_map.html")

✅ 생성: seoul_elderly_map.html
   사고: 1,963건
   승차 TOP: 200, 환승 TOP: 200, 위험 TOP: 200, 사고인접: 200

브라우저에서 열기: open seoul_elderly_map.html
